## 📊 Creación de esquiemas:
#### 📁 Capaz: Bronze, Silver y Gold.

In [0]:
%sql
-- Crear el esquema para la capa Bronze
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

-- Crear el esquema para la capa Silver
CREATE SCHEMA IF NOT EXISTS workspace.silver;

-- Crear el esquema para la capa Gold
CREATE SCHEMA IF NOT EXISTS workspace.gold;


## 📊 Proceso de cargue ingesta de información y almacenamiento en capa bronze:
#### 📁 Paso 1: Se leyó el archivo CSV desde Google Drive.
#### 📁 Paso 2: Se convirtió de ANSI (cp1252) a UTF-8.
#### 📁 Paso 3: Se cargaron todas las columnas como texto para evitar errores de tipos de datos.
#### 📁 Paso 4: Se convirtió el archivo a un DataFrame de Spark.
#### 📁 Paso 5: Se creó el esquema workspace.bronze.
#### 📁 Paso 6: Los datos se almacenaron como tabla Delta convertidos a formato Parquet en: workspace.bronze.movcomercial

**💡 Nota:** El archivo debe estar delimitado por punto y coma (`;`) y tener encabezados en la primera fila.

In [0]:
import pandas as pd
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
# ==========================================
# ID DEL ARCHIVO EN GOOGLE
# ==========================================
file_id = "184dVBWTbhJ6Um64PgmD0364kQvvDQHv0"
# ==========================================
# CONSTRUIR URL DIRECTA
# ==========================================
url = f"https://drive.google.com/uc?id={file_id}"
# ==========================================
# LEER CSV ANSI (cp1252)
# ==========================================
pdf = pd.read_csv(
    url,
    sep=";",
    encoding="cp1252",
    dtype=str,
    low_memory=False
)

# ==========================================
# REEMPLAZAR NULOS
# ==========================================
pdf = pdf.fillna("")
# ==========================================
# CREAR ESQUEMA TODO STRING 
# (Recomendado para capa Bronze)
# ==========================================
schema = StructType([
    StructField(col, StringType(), True)
    for col in pdf.columns
])

# ==========================================
# CREAR DATAFRAME SPARK
# ==========================================
df = spark.createDataFrame(
    pdf.to_dict(orient="records"),
    schema=schema
)

# ==========================================
# MOSTRAR INFORMACIÓN
# ==========================================
display(df)
df.printSchema()
print(f"Total registros: {df.count()}")

# ==========================================
# CREAR ESQUEMA BRONZE SI NO EXISTE
# ==========================================
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

# ==========================================
# GUARDAR TABLA EN FORMATO DELTA
# (Internamente usa archivos parquet)
# ==========================================
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.movcomercial")

print("Tabla workspace.bronze.movcomercial creada correctamente")

# ==========================================
# VALIDAR TABLA CREADA
# ==========================================
df_bronze = spark.table("workspace.bronze.movcomercial")
display(df_bronze)

# ==========================================
# VER DETALLE DEL STORAGE
# ==========================================
spark.sql("""
DESCRIBE DETAIL workspace.bronze.movcomercial
""").display()